<a href="https://colab.research.google.com/github/UW-CTRL/lmc-exercises/blob/main/demos/sqp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install dynamaxsys

In [ ]:
import jax.numpy as jnp
import jax
import numpy as np
import matplotlib.pyplot as plt

from ipywidgets import interact

import cvxpy as cp
import dynamaxsys
import equinox as eqx
from typing import Callable

# Animation of robot and human trajectories using matplotlib.animation
import matplotlib.animation as animation

# To display the animation inline in a Jupyter notebook:
from IPython.display import HTML


Set up dynamics and obstacle function

In [ ]:
wheelbase = 1.0
dt = 0.1
ct_dynamics = dynamaxsys.DynamicallyExtendedSimpleCar(wheelbase)
robot = dynamaxsys.get_discrete_time_dynamics(ct_dynamics, dt)  # robot dynamics
human = dynamaxsys.get_discrete_time_dynamics(ct_dynamics, dt)  # human dynamics


@jax.jit
def obstacle_constraint(state, obstacle, radius):
    return jnp.linalg.norm(state[:2] - obstacle[:2]) - radius


Set up planning parametners

In [ ]:
planning_horizon = 25  # number of timesteps to plan at each iteration
num_time_steps = 30  # number of timesteps to simulate
num_sqp_iterations = 4  # number of SQP iterations to run at each timestep
t = 0.0  # this doesn't affect anything, but a value is needed
radius = 1.0  # minimum collision distance

# limits of systems
v_max = 1.5
v_min = 0.0
acceleration_max = 1.0
acceleration_min = -1.0
steering_max = 0.3
steering_min = -0.3

# noise and variance of human control prediction
human_control_prediction_noise_limit = 0.5
human_control_prediction_variance = 0.5


Set up the inner sqp optimization problem using CVXPY.

In [ ]:
xs = cp.Variable([planning_horizon + 1, robot.state_dim])  # cvx variable for states
us = cp.Variable([planning_horizon, robot.control_dim])  # cvx variable for controls
slack = cp.Variable(1)  # slack variable to make sure the problem is feasible
As = [
    cp.Parameter([robot.state_dim, robot.state_dim]) for _ in range(planning_horizon)
]  # parameters for linearized dynamics
Bs = [
    cp.Parameter([robot.state_dim, robot.control_dim]) for _ in range(planning_horizon)
]  # parameters for linearized dynamics
Cs = [
    cp.Parameter([robot.state_dim]) for _ in range(planning_horizon)
]  # parameters for linearized dynamics

Gs = [
    cp.Parameter([robot.state_dim]) for _ in range(planning_horizon + 1)
]  # parameters for linearized constraints
hs = [
    cp.Parameter(1) for _ in range(planning_horizon + 1)
]  # parameters for linearized constraints

xs_previous = cp.Parameter(
    [planning_horizon + 1, robot.state_dim]
)  # parameter for previous solution
us_previous = cp.Parameter(
    [planning_horizon, robot.control_dim]
)  # parameter for previous solution
initial_state = cp.Parameter([robot.state_dim])  # parameter for current robot state


Setting up the objective

In [ ]:
beta1 = 0.2  # coefficient for control effort
beta2 = 20.0  # coefficient for progress
beta3 = 50.0  # coefficient for trust region
slack_penalty = 1000.0  # coefficient for slack variable
markup = 1.0

objective = (
    beta2 * (xs[-1, 2] ** 2 + xs[-1, 1] ** 2 - 5 * xs[-1, 0])
    + beta3 * (cp.sum_squares(xs - xs_previous) + cp.sum_squares(us - us_previous))
    + slack_penalty * slack**2
)
constraints = [xs[0] == initial_state, slack >= 0]  # initial state and slack constraint
for t in range(planning_horizon):
    objective += beta1 * cp.sum_squares(us[t]) * markup**t
    constraints += [
        xs[t + 1] == As[t] @ xs[t] + Bs[t] @ us[t] + Cs[t]
    ]  # dynamics constraint
    constraints += [
        xs[t, -1] <= v_max,
        xs[t, -1] >= v_min,
        us[t, 0] <= steering_max,
        us[t, 0] >= steering_min,
        us[t, 1] <= acceleration_max,
        us[t, 1] >= acceleration_min,
    ]  # control limit constraints
    constraints += [
        Gs[t] @ xs[t] + hs[t] >= -slack
    ]  # linearized collision avoidance constraint
constraints += [
    xs[planning_horizon, -1] <= v_max,
    xs[planning_horizon, -1] >= v_min,
    Gs[planning_horizon] @ xs[planning_horizon] + hs[planning_horizon] >= 0,
]  # constraints for last planning horizon step
prob = cp.Problem(cp.Minimize(objective), constraints)  # construct problem


In [ ]:
@eqx.filter_jit
def simulate_dynamics(
    dynamics: Callable[[jnp.ndarray, jnp.ndarray, float], jnp.ndarray],
    initial_state: jnp.ndarray,
    control_sequence: jnp.ndarray,
    dt: float,
) -> jnp.ndarray:
    """
    Simulate the discrete-time dynamics over a given time horizon.

    Args:
        dynamics: A callable representing the discrete-time dynamics function.
        initial_state: The initial state of the system.
        control_sequence: A sequence of control inputs for each time step.
        dt: The time step size.

    Returns:
        An array containing the state trajectory over the time horizon.
    """

    def scan_step(state, control):
        x, time = state
        x_next = dynamics(x, control, time)
        return (x_next, time + dt), x_next

    xs_ = jax.lax.scan(scan_step, (initial_state, 0), control_sequence)[1]
    xs = jnp.concatenate([initial_state[None], xs_], axis=0)
    return xs


In [ ]:
# initial states
robot_state = jnp.array([-3.0, -0.0, -0.0, 1.0])  # robot starting state
human_state = jnp.array([-1.0, -2.0, jnp.pi / 2, 1.0])  # human starting state

robot_trajectory = [robot_state]  # list to collect robot's state as it replans
human_trajectory = [human_state]  # list to collect humans's state
robot_control_list = []  # list to collect robot's constrols as it replans
robot_trajectory_list = []  # list to collect robot's planned trajectories

# initial robot planned state and controls
previous_controls = jnp.zeros(
    [planning_horizon, robot.control_dim]
)  # initial guess for robot controls
previous_states = simulate_dynamics(
    robot, robot_state, previous_controls, dt
)  # initial guess for robot states
xs_previous.value = np.array(previous_states)  # set xs_previous parameter value
us_previous.value = np.array(previous_controls)  # set us_previous parameter value

# jit the linearize dynamics and constraint functions to make it run faster
# linearize_dynamics = eqx.filter_jit(lambda states, controls, ti: jax.vmap(dynamaxsys.linearize, [None, 0, 0, None])(lambda s, c, t: robot.discrete_step(s, c, t, dt), states, controls, ti))
linearize_obstacle = eqx.filter_jit(
    lambda states, controls, radius: jax.vmap(
        jax.grad(obstacle_constraint), [0, 0, None]
    )(states, controls, radius)
)

In this simulation, the robot makes a *single* prediction of the human's future trajectory and then makes a plan given that predicted trajectory.

In [ ]:
solver = cp.OSQP
sqp_iteration_list = []
sqp_iteration_control_list = []
key = jax.random.PRNGKey(0)

for t in range(num_time_steps):
    if t % 10 == 0:
        print("timestep: %i" % t)
    initial_state.value = np.array(robot_state)
    # simulate human future trajectory, assuming some noisy behavior
    noisy_human_control = jnp.clip(
        jnp.array(
            jax.random.normal(key, shape=(planning_horizon, human.control_dim))
            * human_control_prediction_variance
        ),
        -human_control_prediction_noise_limit,
        human_control_prediction_noise_limit,
    )
    # simulate the human's future trajectory given the noisy control
    human_future = simulate_dynamics(human, human_state, noisy_human_control, dt)

    # prepare for SQP optimization
    sqp_iterations = [previous_states]  # list to collect robot's states as it replans
    sqp_iteration_control = []  # list to collect robot's controls as it replans
    for i in range(num_sqp_iterations):
        As_value, Bs_value, _, Cs_value = jax.vmap(
            dynamaxsys.linearize, in_axes=(None, 0, 0)
        )(
            robot, previous_states[:-1], previous_controls
        )  # get linearized dynamics for each timestep
        Gs_value = linearize_obstacle(
            previous_states, human_future, radius
        )  # get linearized obstacle constraint
        hs_value = jax.vmap(obstacle_constraint, [0, 0, None])(
            previous_states, human_future, radius
        ) - jax.vmap(jnp.dot, [0, 0])(
            Gs_value, previous_states
        )  # get linearized obstacle constraint

        # set parameters for SQP optimization
        for i in range(planning_horizon):
            As[i].value = np.array(As_value[i])
            Bs[i].value = np.array(Bs_value[i])
            Cs[i].value = np.array(Cs_value[i])
            Gs[i].value = np.array(Gs_value[i])
            hs[i].value = np.array(hs_value[i : i + 1])
        Gs[planning_horizon].value = np.array(Gs_value[planning_horizon])
        hs[planning_horizon].value = np.array(
            hs_value[planning_horizon : planning_horizon + 1]
        )

        # solve the SQP optimization problem
        result = prob.solve(solver=solver)

        # update the robot's state and control
        previous_controls = us.value
        previous_states = simulate_dynamics(robot, robot_state, previous_controls, dt)
        xs_previous.value = np.array(previous_states)
        us_previous.value = np.array(previous_controls)

        # collect the robot's states and controls
        sqp_iterations.append(previous_states)
        sqp_iteration_control.append(previous_controls)

    # stack the robot's states and controls
    sqp_iteration_list.append(jnp.stack(sqp_iterations, axis=-1))
    sqp_iteration_control_list.append(jnp.stack(sqp_iteration_control, axis=-1))

    # get the robot's first control
    robot_control = previous_controls[0]
    robot_control_list.append(robot_control)  # collect the robot's control

    # robot takes a step
    robot_state = robot(robot_state, robot_control)
    robot_trajectory.append(robot_state)  # collect the robot's state
    robot_trajectory_list.append(
        previous_states
    )  # collect the robot's planned trajectory

    # simulate the human's future trajectory (may be different from the predicted trajectory)
    human_random_control = jnp.clip(
        jnp.array(
            np.random.randn(human.control_dim) * human_control_prediction_variance
        ),
        -human_control_prediction_noise_limit,
        human_control_prediction_noise_limit,
    )
    # human states a step
    human_state = human(human_state, human_random_control)
    human_trajectory.append(human_state)  # collect the human's state

robot_trajectory = jnp.stack(robot_trajectory)  # stack the robot's trajectory
human_trajectory = jnp.stack(human_trajectory)  # stack the human's trajectory
robot_controls = jnp.stack(robot_control_list)  # stack the robot's controls

In [ ]:
arr = jnp.stack(sqp_iteration_list)
lower, upper = (
    jnp.stack(
        [
            robot_trajectory[:, :2].min(axis=0),
            human_trajectory[:, :2].min(axis=0),
            arr.min(axis=tuple(i for i in range(arr.ndim) if i != 2))[:2],
        ]
    ).min(axis=0),
    jnp.stack(
        [
            robot_trajectory[:, :2].max(axis=0),
            human_trajectory[:, :2].max(axis=0),
            arr.max(axis=tuple(i for i in range(arr.ndim) if i != 2))[:2],
        ]
    ).max(axis=0),
)
lower, upper

In [ ]:
fig, ax = plt.subplots()


def init():
    ax.clear()
    ax.set_xlim([-4, 2])
    ax.set_ylim([-3, 3])
    # ax.axis("equal")
    ax.grid()
    ax.legend()
    return []


(robot_traj,) = ax.plot([], [], "o-", markersize=3, color="C0")
(robot_planned,) = ax.plot([], [], "o-", markersize=3, color="C2", label="planned")
(human_traj,) = ax.plot([], [], "o-", markersize=3, color="C1")
(sqp_iter,) = ax.plot([], [], "o-", markersize=3, color="C2", alpha=0.2)
robot_point = ax.scatter([], [], s=30, color="C0", label="Robot")
human_point = ax.scatter([], [], s=30, color="C1", label="Human")
circle1 = plt.Circle((0, 0), radius / 2, color="C0", alpha=0.4)
circle2 = plt.Circle((0, 0), radius / 2, color="C1", alpha=0.4)
ax.add_patch(circle1)
ax.add_patch(circle2)


def animate(i):
    ax.clear()
    # Same setup as init, add limits, labels, etc again since ax.clear() wipes them.

    robot_position = robot_trajectory[i, :2]
    human_position = human_trajectory[i, :2]
    circle1 = plt.Circle(robot_position, radius / 2, color="C0", alpha=0.4)
    circle2 = plt.Circle(human_position, radius / 2, color="C1", alpha=0.4)
    ax.add_patch(circle1)
    ax.add_patch(circle2)

    ax.plot(
        robot_trajectory[:, 0], robot_trajectory[:, 1], "o-", markersize=3, color="C0"
    )
    ax.plot(
        robot_trajectory_list[i][:, 0],
        robot_trajectory_list[i][:, 1],
        "o-",
        markersize=3,
        color="C2",
        label="planned",
    )
    ax.plot(
        human_trajectory[:, 0], human_trajectory[:, 1], "o-", markersize=3, color="C1"
    )
    ax.plot(
        sqp_iteration_list[i][:, 0],
        sqp_iteration_list[i][:, 1],
        "o-",
        markersize=3,
        color="C2",
        alpha=0.2,
    )
    ax.scatter(
        robot_trajectory[i : i + 1, 0],
        robot_trajectory[i : i + 1, 1],
        s=30,
        color="C0",
        label="Robot",
    )
    ax.scatter(
        human_trajectory[i : i + 1, 0],
        human_trajectory[i : i + 1, 1],
        s=30,
        color="C1",
        label="Human",
    )
    ax.set_title(
        "heading=%.2f velocity=%.2f" % (robot_trajectory[i, 2], robot_trajectory[i, 3])
    )
    ax.legend()
    ax.axis("equal")
    eps = 4
    ax.set_xlim([lower[0] - eps, upper[0] + eps])
    ax.set_ylim([lower[1] - eps, upper[1] + eps])

    ax.grid()


ani = animation.FuncAnimation(
    fig, animate, frames=num_time_steps, interval=150, blit=False, repeat=True
)


HTML(ani.to_jshtml())

In [ ]:
# plotting
@interact(i=(0, num_time_steps - 1))
def plot(i):
    fig, axs = plt.subplots(1, 2, figsize=(18, 8))

    ax = axs[0]
    # ax.clear()
    robot_position = robot_trajectory[i, :2]
    human_position = human_trajectory[i, :2]
    circle1 = plt.Circle(robot_position, radius / 2, color="C0", alpha=0.4)
    circle2 = plt.Circle(human_position, radius / 2, color="C1", alpha=0.4)
    ax.add_patch(circle1)
    ax.add_patch(circle2)
    # ax.plot(human_samples[i,:,:,0].T, human_samples[i,:,:,1].T, "o-", alpha=0.1, markersize=2, color='C1')
    ax.plot(
        robot_trajectory[:, 0], robot_trajectory[:, 1], "o-", markersize=3, color="C0"
    )
    ax.plot(
        robot_trajectory_list[i][:, 0],
        robot_trajectory_list[i][:, 1],
        "o-",
        markersize=3,
        color="C2",
        label="planned",
    )

    ax.plot(
        human_trajectory[:, 0], human_trajectory[:, 1], "o-", markersize=3, color="C1"
    )
    ax.plot(
        sqp_iteration_list[i][:, 0],
        sqp_iteration_list[i][:, 1],
        "o-",
        markersize=3,
        color="C2",
        alpha=0.2,
    )
    ax.scatter(
        robot_trajectory[i : i + 1, 0],
        robot_trajectory[i : i + 1, 1],
        s=30,
        color="C0",
        label="Robot",
    )
    ax.scatter(
        human_trajectory[i : i + 1, 0],
        human_trajectory[i : i + 1, 1],
        s=30,
        color="C1",
        label="Human",
    )
    ax.grid()
    ax.legend()
    ax.axis("equal")
    eps = 1
    ax.set_xlim([lower[0] - eps, upper[0] + eps])
    ax.set_ylim([lower[1] - eps, upper[1] + eps])


    ax.set_title(
        "heading=%.2f velocity=%.2f" % (robot_trajectory[i, 2], robot_trajectory[i, 3])
    )

    ax = axs[1]
    plt.plot(robot_controls)
    plt.scatter([i], robot_controls[i : i + 1, 0], label="Steering")
    plt.scatter([i], robot_controls[i : i + 1, 1], label="Acceleration")
    ax.plot(robot_trajectory[:, -1], "o-", markersize=3, color="C2", label="Velocity")

    ax.hlines(steering_max, 0, num_time_steps, color="C0", alpha=0.4, linewidth=5)
    ax.hlines(steering_min, 0, num_time_steps, color="C0", alpha=0.4, linewidth=5)
    ax.hlines(acceleration_max, 0, num_time_steps, color="C1", alpha=0.4, linewidth=5)
    ax.hlines(acceleration_min, 0, num_time_steps, color="C1", alpha=0.4, linewidth=5)
    ax.hlines(v_max, 0, num_time_steps, color="C2", alpha=0.4, linewidth=5)
    ax.hlines(v_min, 0, num_time_steps, color="C2", alpha=0.4, linewidth=5)

    ax.legend()
    ax.grid()